# Learning Decision Trees by Inventing Them

**Philosophy:** You won't be taught — you will *discover*. Every step builds on the last. Trust the process.

# PART 1: What Is Impurity?

## Exercise 1.1 — Variance

For analogy if we think about how impure a glass of water is, that means how many unwanted elements are in a glass of water, correct?
But in computer science we talk about numbers, For numbers, we don't have dirt or chemicals. Instead, we think about how mixed up the values are.

Consider these two groups:

In [23]:
group_a = [10, 10, 10, 10, 10]

group_b = [2, 8, 15, 22, 30]


"How similar are the numbers in each group?"

In [24]:
group_c = [10, 11, 9, 10, 10]

Is this pure or impure?

<details>
  <summary>Conclusion</summary>

So we can think of purity as a spectrum:

```
[10, 10, 10, 10, 10]        -> Very Pure
[10, 11, 9, 10, 10]         -> Mostly Pure
[2, 8, 15, 22, 30]          -> Impure
```

We can visually tell which group is impure. But how can a computer measure impurity? Computers need numbers.
</details>

We said that a pure group contains similar values and an impure group contains very different values.

Question: How can we measure that with a function?

<details>
  <summary>Answer</summary>

Let's find some "central" value and see how far every number is from it.
> The natural central value is the mean.
</details>

In [27]:
def variance(values):
    
    if len(values) == 0:
        return 0
    
    mean = sum(values) / len(values)

    total_squared_distance = 0

    for value in values:
        total_squared_distance += (value - mean) ** 2
    
    variance = total_squared_distance / len(values)

    return round(variance, 2)

In [28]:
group_a = [10, 10, 10, 10, 10]
print(variance(group_a))

group_b = [2, 8, 15, 22, 30]
print(variance(group_b))

group_c = [10, 11, 9, 10, 10]
print(variance(group_c))

group_d = [0, 5, 17, 23, 45]
print(variance(group_d))

group_e = [1, 2, 3, 4, 10, 100]
print(variance(group_e))

0.0
98.24
0.4
249.6
1288.33


We noticed that variance does **not** have a fixed upper bound.

### Can we force it into 0-1?

Yes, using this formula: impurity_score = variance / (1 + variance)

In [30]:
group_a = [10, 10, 10, 10, 10]
impurity_score = round(variance(group_a) / (1 + variance(group_a)), 2)
print(impurity_score)

group_b = [2, 8, 15, 22, 30]
impurity_score = round(variance(group_b) / (1 + variance(group_b)), 2)
print(impurity_score)

group_c = [10, 11, 9, 10, 10]
impurity_score = round(variance(group_c) / (1 + variance(group_c)), 2)
print(impurity_score)

group_d = [0, 5, 17, 23, 45]
impurity_score = round(variance(group_d) / (1 + variance(group_d)), 2)
print(impurity_score)

group_e = [1, 2, 3, 4, 10, 100]
impurity_score = round(variance(group_e) / (1 + variance(group_e)), 2)
print(impurity_score)

0.0
0.99
0.29
1.0
1.0


But by converting `variance` into `impurity score` we are eventually **loosing information**. Thats the reason actual **Decission tree regression** doesn't use `impurity score` it uses the `variance itself`.

## Exercise 1.2 — Weighted Total Variance

We have learned how to measure the impurity(variance) of a group of numbers.

Now imagine we have an impure group:
```
values = [10, 9, 11, 8, 10, 1]
```
Its impurity is fairly high because one value (1) is very different from the others.

Question: Can we make this less impure somehow?

<details>
  <summary>Answer</summary>

By splitting them into 2 we can make it less impure. Just like removing the impure elements from impure water to make it pure.
</details>


In [48]:
values = [10, 9, 11, 8, 10, 1]
print(variance(values))

left  = [10, 9, 11, 8, 10]
print(variance(left))
right = [1]
print(variance(right))

11.14
1.04
0.0


Now `left` has very small variance (less impure) and `right` is exactly 0 because a single value is perfectly pure.

We've turned one impure group into two much purer groups.


### The New Problem

When we split a list into two parts, we get two impurity values:
`left variance` and `right variance`

But a Decision Tree needs one score to decide whether a split is good.
How should we combine them?

**Consider:**
```
left  = [100 numbers]
right = [2 numbers]
```

Should both groups contribute equally?

Probably not.
The larger group should matter more.

### Your Challenge

Invent a formula for the total impurity of a split.

Think about these questions:
- If one group has 100 items and another has 2 items, should they have equal influence?
- If one group is empty, what should happen?
- If both groups have the same impurity and the same size, what should the answer be?
- What mathematical concept combines values while accounting for size?

> Hint: Weighted average.

In [49]:
def total_variance(left, right, impurity_fnc):
    total_size = len(left) + len(right)

    left_weight = len(left) / total_size
    right_weight = len(right) / total_size

    total_variance = left_weight * impurity_fnc(left) + right_weight * impurity_fnc(right)

    return round(total_variance, 2)

Test it with:

- total_variance([10,10,10], [0,0,0]) # perfectly separated
- total_variance([10,0,9], [0,10,2]) # each group is mixed 
- total_variance([10, 9, 11, 8, 10, 11, 12, 12, 9, 2], [1]) # unequal sizes
- total_variance([10,0,9], []) # One group is empty

In [50]:
print(total_variance([10,10,10], [0,0,0], variance))
print(total_variance([10,0,9], [0,10,2], variance))
print(total_variance([10, 9, 11, 8, 10, 11, 12, 12, 9, 2], [1], variance))
print(total_variance([10,0,9], [], variance))

0.0
19.45
6.95
20.22


In [34]:
variance([10, 9, 11, 8, 10, 11, 12, 12, 9, 2])

7.64

In [35]:
variance([1])

0.0

This sets up the next question perfectly:

- We know the impurity before the split.
- We know the weighted impurity after the split.

Question: We split the data. But how do we know whether the split was actually good?

<details>
  <summary>Answer</summary>

By looking at how much variance we reduce by spliting.
```
improvement = old_variance - new_variance

or

variance_reduction = parent_variance - child_variance
```
</details>

## Exercise 1.3 — Variance Reduction

In [36]:
values = [10, 9, 11, 8, 10, 1]
parent_variance = variance(values)
print(parent_variance)

11.14


In [38]:
left = [10, 9, 11, 8, 10]
right = [1]
child_variance = total_variance(left, right, variance)
print(child_variance)

0.87


In [39]:
variance_reduction = round(parent_variance - child_variance, 2)
print(variance_reduction)

10.27


In [40]:
left = [10, 9, 11]
right = [8, 10, 1]
child_variance = total_variance(left, right, variance)
print(child_variance)

variance_reduction = round(parent_variance - child_variance, 2)
print(variance_reduction)

7.78
3.36


Think of variance as "messiness."

```
Parent node:
Messiness = 11.14

After split:
Messiness = 0.84
```

We removed: 11.14 - 0.84 = **10.27** units of messiness.

> The more messiness we remove, the better the split. | more **variance_reduction is better**.

# PART 2: Helper concepts for Decision Tree

## Exercise 2.1 — Best Split

Okay, but real datasets have many features. How does the tree know which feature to split on?"

Let's Create a Tiny Dataset

Suppose we're trying to predict house price.
| Size | Bedrooms | Price |
|------|----------|-------|
| 500  | 1 | 50   |
| 600  | 1 | 55   |
| 900  | 3 | 65   |
| 1500 | 3 | 150  | 
| 1600 | 3 | 160  |
| 1700 | 4 | 170  |

In [51]:
y = [50, 55, 65, 150, 160, 170]
parent_variance = variance(y)
print(parent_variance)

2722.22


#### Lets split on **Feature 1: Size** at Size < 1000

In [52]:
size_left = [50, 55, 65]
size_right = [150, 160, 170]
child_variance = total_variance(size_left, size_right, variance)
print(child_variance)

52.78


In [53]:
size_variance_reduction = round(parent_variance - child_variance, 2)
print(size_variance_reduction)

2669.44


#### Lets split on **Feature 2: Bedrooms** at Bedrooms < 3

In [54]:
left_bedroom = [50,55]
right_bedroom = [65,150,160,170]
child_variance = total_variance(left_bedroom, right_bedroom, variance)
print(child_variance)

1163.54


In [55]:
bedroom_variance_reduction = round(parent_variance - child_variance, 2)
print(bedroom_variance_reduction)

1558.68


In [ ]:
# root: [50, 55, 65, 150, 160, 170] Condition size < 1000
#                   |
#left:[50, 55, 65] condition bedrrom<=2         right: [150, 160, 170]  
#        |
# leaf left node: [50] leaf right node: [55, 65]   -> condition size <=2 
#   

In [ ]:
# to predict a = [50, 2] -> [size, bedroom]
# root: [50, 55, 65, 150, 160, 170] -> condition: size<1000
# check: a[0] =50 < 1000: True => move to left
# left: [50, 55, 65] -> condition: None => leaf node
# predict: average(values in leaf node) => 50+55+65/3 => print((50+55+65)/3)
print((50+55+65)/3)

56.666666666666664


Which feature should we choose?

*The feature giving the largest variance reduction. Here that is Size of the House*

What if there are 100 features?

*We do exactly the same thing.*

```
Feature 1
    Try every possible split
    Keep the best

Feature 2
    Try every possible split
    Keep the best

Feature 3
    Try every possible split
    Keep the best

...

Feature 100
    Try every possible split
    Keep the best

Then compare all of them.
Choose the split with the highest variance reduction.
```

## Exercise 2.2 — Find Best Split Automatically

lets write a function that will give use best split with threshold autoimatically

In [ ]:
def find_best_split(X, y, variance_fnc):
    parent_variance = variance_fnc(y)

    best_feature = None
    best_threshold = None
    best_variance_reduction = -float("inf")

    n_features = len(X[0])

    # Try every feature
    for feature_idx in range(n_features):

        # All unique values in this feature
        feature_values = set(row[feature_idx] for row in X)

        # Try every value as a threshold
        for threshold in feature_values:

            left_y = []
            right_y = []

            for row, target in zip(X, y):

                if row[feature_idx] <= threshold:
                    left_y.append(target)
                else:
                    right_y.append(target)

            # Skip useless splits
            if len(left_y) == 0 or len(right_y) == 0:
                continue

            children_variance = total_variance(
                left_y,
                right_y,
                variance_fnc
            )

            variance_reduction = (
                parent_variance
                - children_variance
            )

            if variance_reduction > best_variance_reduction:
                best_variance_reduction = variance_reduction
                best_feature = feature_idx
                best_threshold = threshold

    return {
        "feature": best_feature,
        "threshold": best_threshold,
        "variance_reduction": best_variance_reduction
    }

In [60]:
X = [
    [500, 1],
    [600, 1],
    [900, 3],
    [1500, 3],
    [1600, 3],
    [1700, 4]
]

y = [50, 55, 65, 150, 160, 170]

print(find_best_split(X, y, variance))

{'feature': 0, 'threshold': 900, 'variance_reduction': 2669.4399999999996}


Let us take another feature: Distance from City Center (best fit)

- **Value = Meaning**
- **1 = Downtown**
- **2 = Suburban**
- **3 = Rural/Far Away**

In [61]:
X = [
    [500, 1, 3],
    [600, 1, 1],
    [900, 3, 2],
    [1500, 3, 1],
    [1600, 3, 3],
    [1700, 4, 2]
]

y = [50, 150, 65, 155, 90, 170]

print(find_best_split(X, y, variance))

{'feature': 2, 'threshold': 2, 'variance_reduction': 938.8899999999999}


A decision tree is made of **nodes**. There are two kinds:

- **Boundary node:** Has a rule — "if feature X ≤ value, go left; else go right." Has two children.
- **Leaf node:** Has final answers — "list of numbers till this point eg: [100, 120, 109, 117]"

So far we've built a function that can find the best split for a single node.

A Decision Tree is simply the same process repeated recursively:
- Find the best split.
- Create a left child.
- Create a right child.
- Repeat on both children until stopping conditions are met.

What are these stopping conditions coz without proper stopping conditions we are just mapping all the samples/training data that will end up as overfitting coz model just memorized all the samples provided.

If we keep splitting forever, won't the tree become huge?

This is where hyperparameters come in.

## Exercise 2.3 — max_depth

How many questions should the tree be allowed to ask?

Consider:
```
Is Size <= 1000?
├── Yes
│   ├── Is Bedrooms <= 2?
│   │   ├── Yes
│   │   └── No
│   └── ...
└── No
```

Every level is another question.

The depth of a tree is simply the maximum number of questions from the root to a leaf.

Example:
```
Depth = 1

        Root
       /    \
    Leaf   Leaf

Depth = 2

         Root
        /    \
      Node   Leaf
     /   \
  Leaf  Leaf
```

If we allow: max_depth = 2

> the tree can only ask 2 questions before stopping.

If we allow: max_depth = 20

> the tree can become much more complex.

**Small depth:**
- Simple tree
- Less memorization
- May miss patterns

**Large depth:**
- Complex tree
- Can learn detailed patterns
- May overfit

## Exercise 2.4 — min_samples_split

Should we split a node that contains only 2 samples?

Suppose we reach:
```
[150, 160]
```

A split could produce:
```
[150]
[160]
```

Both groups are perfectly pure.
> Variance: 0,0 

Looks amazing! But did we actually learn anything?

Not really—we just memorized the training data.

min_samples_split prevents this.
```
min_samples_split = 5

means: A node must contain at least 5 samples before we are allowed to split it.
```

So: [10, 12, 14]

contains only 3 samples.

No splitting allowed. It automatically becomes a leaf.

> When we reach a leaf node and got more than 1 value, return average of all these values as prediction

**Small value of min_sample_split:**
- More splitting
- More complex tree
- Higher overfitting risk

**Large value of min_sample_split:**
- Less splitting
- Simpler tree
- Higher underfitting risk

#### These hyperparameters act like brakes:

- max_depth: limits how many questions the tree can ask.

- min_samples_split: limits when the tree is allowed to ask another question.

# PART 3: Build Decision Tree

## Exercise 3.1 — Node Class

let's create Node Class, A node needs to remember:
- which feature to split on
- which threshold to use
- left child
- right child
- prediction value (for leaf nodes)


In [62]:
class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

## Exercise 3.2 — Tree Class

Tree class will have:
- Hyperparameters
- Root Node
- Methods like: fit, predict, and helper methods

In [63]:
class DecisionTreeRegressor:

    def __init__(
        self,
        max_depth,
        min_samples_split
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    def variance(self, values):
        if len(values) == 0:
            return 0
        
        mean = sum(values) / len(values)
        total_squared_distance = 0

        for value in values:
            total_squared_distance += (value - mean) ** 2

        return total_squared_distance / len(values)

    def total_variance(self,left, right):
        total_size = len(left) + len(right)

        left_weight = len(left) / total_size
        right_weight = len(right) / total_size

        return left_weight * self.variance(left) + right_weight * self.variance(right)

    def find_best_split(self, X, y):
        parent_variance = self.variance(y)
        best_feature = None
        best_threshold = None
        best_gain = -float("inf")

        n_features = len(X[0])
        for feature_idx in range(n_features):

            thresholds = sorted(
                set(row[feature_idx] for row in X)
            )

            for threshold in thresholds:
                left_y = []
                right_y = []

                for row, target in zip(X, y):
                    if row[feature_idx] <= threshold:
                        left_y.append(target)
                    else:
                        right_y.append(target)

                if not left_y or not right_y:
                    continue

                child_variance = (self.total_variance(left_y,right_y))

                gain = (parent_variance - child_variance)

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return (best_feature,best_threshold)
    
    def build_tree(self, X, y, depth=0):
        # Stop if too few samples
        if len(y) < self.min_samples_split:
            return Node(value=sum(y) / len(y))

        # Stop if maximum depth reached
        if depth >= self.max_depth:
            return Node(value=sum(y) / len(y))

        # Stop if node is perfectly pure
        if self.variance(y) == 0:
            return Node(value=y[0])

        # Find the best split
        feature, threshold = self.find_best_split(X, y)
        # No valid split found
        if feature is None:
            return Node(value=sum(y) / len(y))

        # Split the data
        left_X = []
        left_y = []
        right_X = []
        right_y = []

        for row, target in zip(X, y):
            if row[feature] <= threshold:
                left_X.append(row)
                left_y.append(target)
            else:
                right_X.append(row)
                right_y.append(target)

        # Recursively build children
        left_child = self.build_tree(
            left_X,
            left_y,
            depth + 1
        )

        right_child = self.build_tree(
            right_X,
            right_y,
            depth + 1
        )

        # Return internal node
        return Node(
            feature=feature,
            threshold=threshold,
            left=left_child,
            right=right_child
        )
    
    def fit(self, X, y):
        self.root = self.build_tree(X, y)
        
    def predict_one(self,row,node):
        if node.is_leaf():
            return node.value

        if row[node.feature] <= node.threshold:
            return self.predict_one(
                row,
                node.left
            )

        return self.predict_one(
            row,
            node.right
        )

    def predict(self, X):
        predictions = []

        for row in X:
            predictions.append(
                self.predict_one(
                    row,
                    self.root
                )
            )

        return predictions

In [64]:
def root_mean_squared_error(y_true, y_pred):
    n = len(y_true)
    mse = sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred)) / n
    return round(mse ** 0.5, 2)

In [72]:
X = [
    [500, 1],
    [600, 1],
    [700, 2],
    [1500, 3],
    [1600, 3],
    [1700, 4]
]

y = [50, 55, 65, 150, 160, 170]

tree = DecisionTreeRegressor(
    max_depth=2,
    min_samples_split=2
)

tree.fit(X, y)

print(
    tree.predict([
        [500, 1],
        [1600, 3]
    ])
)

[52.5, 165.0]


In [73]:

print(
    tree.predict([
        [550, 1],
        [1650, 3]
    ])
)

[52.5, 165.0]


In [74]:
train_error = root_mean_squared_error(y, tree.predict(X))
print("Train RMSE:", train_error)

test_error = root_mean_squared_error([52, 165], tree.predict([[550, 1], [1650, 3]]))
print("Test RMSE:", test_error)

Train RMSE: 3.23
Test RMSE: 0.35


In [17]:
X = [
    [500, 1, 3],
    [600, 1, 1],
    [900, 3, 2],
    [1500, 3, 1],
    [1600, 3, 3],
    [1700, 4, 2]
]

y = [50, 90, 110, 150, 160, 180]

tree1 = DecisionTreeRegressor(
    max_depth=2,
    min_samples_split=2
)

tree1.fit(X, y)

print(
    tree1.predict([
        [500, 1, 3],
        [1600, 3, 3]
    ])
)

[50.0, 155.0]


In [18]:
print(
    tree1.predict([
        [550, 1, 1],
        [1650, 3, 2]
    ])
)

[100.0, 180.0]


In [19]:
train_error = root_mean_squared_error(y, tree1.predict(X))
print("Train RMSE:", train_error)
test_error = root_mean_squared_error([80, 170], tree1.predict([[550, 1, 1], [1650, 3, 2]]))
print("Test RMSE:", test_error)

Train RMSE: 6.45
Test RMSE: 15.81
